## 1. Imports and Setup

In [4]:
import pandas as pd
from pathlib import Path

AUDIOSET_DIR = Path("data/audio/AudioSet")
AUDIO_OUT    = AUDIOSET_DIR / "clips"
AUDIO_OUT.mkdir(parents=True, exist_ok=True)

# Load metadata
balanced = pd.read_csv(
    AUDIOSET_DIR / "balanced_train_segments.csv",
    skiprows=3,
    header=None,
    names=['ytid', 'start', 'end', 'labels'],
    quotechar='"',
    skipinitialspace=True,
    on_bad_lines='skip'
)

eval_seg = pd.read_csv(
    AUDIOSET_DIR / "eval_segments.csv",
    skiprows=3,
    header=None,
    names=['ytid', 'start', 'end', 'labels'],
    quotechar='"',
    skipinitialspace=True,
    on_bad_lines='skip'
)

classes = pd.read_csv(AUDIOSET_DIR / "class_labels_indices.csv")

print(f"Balanced train segments: {len(balanced)}")
print(f"Eval segments:           {len(eval_seg)}")
print(f"Total classes:           {len(classes)}")
print(f"\nSample rows:")
balanced.head()

Balanced train segments: 22160
Eval segments:           20371
Total classes:           527

Sample rows:


,ytid,start,end,labels
0,--PJHxphWEs,30.0,40.0,"/m/09x0r,/t/dd00088"
1,--ZhevVpy1s,50.0,60.0,/m/012xff
2,--aE2O5G5WE,0.0,10.0,"/m/03fwl,/m/04rlf,/m/09x0r"
3,--aO5cdqSAg,30.0,40.0,"/t/dd00003,/t/dd00005"
4,--aaILOrkII,200.0,210.0,"/m/032s66,/m/073cg4"


## 2. Find AudioSet IDs for Our 10 CIFAR-10 Classes

In [8]:
# AudioSet ontology IDs for our 10 CIFAR-10 classes
CIFAR10_AUDIOSET_MAP = {
    'airplane'   : ['/m/0cmf2'],
    'automobile' : ['/m/0k4j'],
    'bird'       : ['/m/015p6'],
    'cat'        : ['/m/01yrx'],
    'deer'       : None,
    'dog'        : ['/m/068hy'],
    'frog'       : ['/m/09ld4'],
    'horse'      : ['/m/03k3r'],
    'ship'       : ['/m/06q74', '/m/019jd'],  # ship + boat/water vehicle
    'truck'      : ['/m/07r04'],
}

CIFAR10_CLASSES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

CIFAR10_LABEL_MAP = {c: i for i, c in enumerate(CIFAR10_CLASSES)}

# Verify IDs exist in class_labels_indices
print("Verifying AudioSet IDs:")
print("-" * 45)
for cifar_class, ids in CIFAR10_AUDIOSET_MAP.items():
    if ids:
        for mid in ids:
            match = classes[classes['mid'] == mid]
            if len(match) > 0:
                print(f"{cifar_class:<15} → {mid}  ✅  {match['display_name'].values[0]}")
            else:
                print(f"{cifar_class:<15} → {mid}  ❌  NOT FOUND")
    else:
        print(f"{cifar_class:<15} → None  ❌  NEEDS SYNTHESIS")

Verifying AudioSet IDs:
---------------------------------------------
airplane        → /m/0cmf2  ✅  Fixed-wing aircraft, airplane
automobile      → /m/0k4j  ✅  Car
bird            → /m/015p6  ✅  Bird
cat             → /m/01yrx  ✅  Cat
deer            → None  ❌  NEEDS SYNTHESIS
dog             → /m/068hy  ✅  Domestic animals, pets
frog            → /m/09ld4  ✅  Frog
horse           → /m/03k3r  ✅  Horse
ship            → /m/06q74  ✅  Ship
ship            → /m/019jd  ✅  Boat, Water vehicle
truck           → /m/07r04  ✅  Truck


## 3. Filter Segments for Our Classes

In [9]:
# Combine balanced train and eval
all_segments = pd.concat([balanced, eval_seg], ignore_index=True)
print(f"Total segments: {len(all_segments)}")

MAX_PER_CLASS = 200

filtered_rows = []

for cifar_class, ids in CIFAR10_AUDIOSET_MAP.items():
    if ids is None:
        continue

    # Find segments containing any of our target IDs
    matches = all_segments[
        all_segments['labels'].apply(
            lambda x: any(mid in str(x) for mid in ids)
        )
    ].copy()

    # Cap at MAX_PER_CLASS
    matches = matches.head(MAX_PER_CLASS)
    matches['cifar_class'] = cifar_class
    matches['cifar_label'] = CIFAR10_LABEL_MAP[cifar_class]

    filtered_rows.append(matches)
    print(f"{cifar_class:<15} → {len(matches)} segments found")

download_df = pd.concat(filtered_rows, ignore_index=True)
print(f"\nTotal segments to download: {len(download_df)}")
download_df.to_csv(AUDIOSET_DIR / "download_list.csv", index=False)
print("Saved download_list.csv")

Total segments: 42531
airplane        → 127 segments found
automobile      → 200 segments found
bird            → 200 segments found
cat             → 200 segments found
dog             → 200 segments found
frog            → 123 segments found
horse           → 163 segments found
ship            → 200 segments found
truck           → 175 segments found

Total segments to download: 1588
Saved download_list.csv


In [7]:
# Search correct IDs from the classes dataframe
search_terms = ['frog', 'horse', 'ship', 'boat', 'vessel', 'insect']
for term in search_terms:
    matches = classes[classes['display_name'].str.lower().str.contains(term)]
    if len(matches) > 0:
        print(f"\n--- {term} ---")
        print(matches[['mid', 'display_name']].to_string())


--- frog ---
          mid display_name
132  /m/09ld4         Frog

--- horse ---
         mid display_name
87  /m/03k3r        Horse

--- ship ---
          mid            display_name
302  /m/0hsrw  Sailboat, sailing ship
305  /m/06q74                    Ship

--- boat ---
           mid            display_name
301   /m/019jd     Boat, Water vehicle
302   /m/0hsrw  Sailboat, sailing ship
303  /m/056ks2   Rowboat, canoe, kayak
304  /m/02rlv9    Motorboat, speedboat

--- insect ---
          mid display_name
126  /m/03vt0       Insect


## 4. Download Audio Clips

In [ ]:
import subprocess
import os
import time

SAMPLE_RATE = 16000
failed      = []
downloaded  = []

print(f"Starting download of {len(download_df)} clips...")
print("This will take a while — some clips may fail if video is unavailable\n")

for i, row in download_df.iterrows():
    ytid       = row['ytid'].strip()
    start      = float(row['start'])
    end        = float(row['end'])
    duration   = end - start
    cls        = row['cifar_class']
    label      = row['cifar_label']

    out_dir    = AUDIO_OUT / cls
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file   = out_dir / f"{ytid}_{int(start)}.wav"

    # Skip if already downloaded
    if out_file.exists():
        downloaded.append(str(out_file))
        continue

    url = f"https://www.youtube.com/watch?v={ytid}"

    cmd = [
        "yt-dlp",
        "--quiet",
        "--no-warnings",
        "-x",
        "--audio-format", "wav",
        "--downloader-args", f"ffmpeg:-ss {start} -t {duration}",
        "--postprocessor-args", f"ffmpeg:-ar {SAMPLE_RATE} -ac 1",
        "-o", str(out_file),
        url
    ]

    try:
        result = subprocess.run(cmd, timeout=60, 
                                capture_output=True, text=True)
        if out_file.exists():
            downloaded.append(str(out_file))
        else:
            failed.append(ytid)
    except subprocess.TimeoutExpired:
        failed.append(ytid)
    except Exception as e:
        failed.append(ytid)

    # Progress every 10 clips
    total_done = len(downloaded) + len(failed)
    if total_done % 10 == 0:
        print(f"  Progress: {total_done}/{len(download_df)} — "
              f"Downloaded: {len(downloaded)} — "
              f"Failed: {len(failed)}")

    time.sleep(0.5)  # small delay to avoid rate limiting

print(f"\n{'='*50}")
print(f"Download complete")
print(f"Successfully downloaded: {len(downloaded)}")
print(f"Failed / unavailable:    {len(failed)}")
print(f"{'='*50}")

## 5. Verify Downloads Per Class

In [ ]:
print("Downloaded clips per class:")
print("-" * 35)
total = 0
for cifar_class in CIFAR10_CLASSES:
    if cifar_class == 'deer':
        print(f"{'deer':<15} → synthesis (pending)")
        continue
    cls_dir = AUDIO_OUT / cifar_class
    count   = len(list(cls_dir.glob("*.wav"))) if cls_dir.exists() else 0
    total  += count
    flag    = " ⚠️" if count < 50 else ""
    print(f"{cifar_class:<15} → {count} clips{flag}")

print(f"\nTotal downloaded: {total}")
print(f"Failed IDs saved for reference: {len(failed)}")

# Save failed list for reference
with open(AUDIOSET_DIR / "failed_downloads.txt", "w") as f:
    f.write("\n".join(failed))
print("Failed IDs saved to failed_downloads.txt")